# P9 — PEFT / LoRA / QLoRA Claims Processing
This Kaggle-friendly notebook prepares and audits an instruction dataset. GPU fine-tuning is an optional extension and depends on available accelerator memory and model access.

In [64]:
!pip -q install pandas scikit-learn

In [65]:
from pathlib import Path
import json
import pandas as pd

# Correct Kaggle dataset path
BASE_DIR = Path(
    "/kaggle/input/datasets/babblu123/"
    "real-world-ml-genai-kaggle-complete/"
    "real_world_ml_genai_lab"
)

DATA_DIR = BASE_DIR / "datasets"

print("Dataset directory:", DATA_DIR)
print("Directory exists:", DATA_DIR.exists())

claims_path = DATA_DIR / "p9_claims.jsonl"

if not claims_path.exists():
    raise FileNotFoundError(
        f"Claims file not found: {claims_path}\n"
        f"Available files: {[p.name for p in DATA_DIR.iterdir()]}"
    )

# Load JSONL records
records = [
    json.loads(line)
    for line in claims_path.read_text(encoding="utf-8").splitlines()
    if line.strip()
]

df = pd.DataFrame(records)

print("Total claims:", len(df))
print("Columns:", df.columns.tolist())

display(df.head())

Dataset directory: /kaggle/input/datasets/babblu123/real-world-ml-genai-kaggle-complete/real_world_ml_genai_lab/datasets
Directory exists: True
Total claims: 500
Columns: ['claim_id', 'claim_type', 'claim_amount', 'narrative', 'expert_decision']


,claim_id,claim_type,claim_amount,narrative,expert_decision
0,CLM00001,auto,588.24,Synthetic auto claim reporting an incident wit...,approve
1,CLM00002,property,6118.08,Synthetic property claim reporting an incident...,request_documents
2,CLM00003,auto,5202.87,Synthetic auto claim reporting an incident wit...,request_documents
3,CLM00004,property,3383.62,Synthetic property claim reporting an incident...,approve
4,CLM00005,auto,2576.50,Synthetic auto claim reporting an incident wit...,approve


In [66]:
print("Rows:",len(df)); display(df["expert_decision"].value_counts())
# Basic de-identification safeguard: only synthetic IDs and narratives in this dataset.
required={"claim_type","claim_amount","narrative","expert_decision"}
assert required.issubset(df.columns)

Rows: 500


expert_decision
approve              328
request_documents    106
review                66
Name: count, dtype: int64

In [67]:
# Convert to chat-style JSONL for instruction tuning.
out=[]
for r in records:
    out.append({"messages":[
        {"role":"system","content":"Classify the synthetic insurance claim using the allowed decisions: approve, request_documents, review. Do not invent facts."},
        {"role":"user","content":f"Claim type: {r['claim_type']}\nAmount: {r['claim_amount']}\nNarrative: {r['narrative']}"},
        {"role":"assistant","content":r["expert_decision"]}
    ]})
train_path="/kaggle/working/claims_instruction.jsonl"
with open(train_path,"w") as f:
    for item in out: f.write(json.dumps(item)+"\n")
print("Wrote",train_path,"records:",len(out))

Wrote /kaggle/working/claims_instruction.jsonl records: 500


## Optional QLoRA training
For a real experiment, select a license-compatible open-weight model, enable Kaggle GPU, install compatible `transformers`, `peft`, `trl`, `bitsandbytes`, and `accelerate`, then configure 4-bit quantization and LoRA adapters. Model downloads may require internet and approval; do not upload real PII or sensitive claims. Compare base vs adapter on held-out data.